In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, regexp_extract, current_timestamp, lit, abs
from pyspark.sql.types import IntegerType, LongType
import os
import shutil
import glob

# Initialize Spark
spark = SparkSession.builder \
    .appName("Silver_Layer_Year_Based_Splitting") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "10") \
    .getOrCreate()

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def remove_comment_column(df, table_name):
    """Remove comment column from dataframe"""
    if 'p_comment' in df.columns:
        df = df.drop('p_comment')
        print(f"✓ Removed p_comment from {table_name}")
    if 'c_comment' in df.columns:
        df = df.drop('c_comment')
        print(f"✓ Removed c_comment from {table_name}")
    if 'o_comment' in df.columns:
        df = df.drop('o_comment')
        print(f"✓ Removed o_comment from {table_name}")
    if 'l_comment' in df.columns:
        df = df.drop('l_comment')
        print(f"✓ Removed l_comment from {table_name}")
    if 's_comment' in df.columns:
        df = df.drop('s_comment')
        print(f"✓ Removed s_comment from {table_name}")
    if 'n_comment' in df.columns:
        df = df.drop('n_comment')
        print(f"✓ Removed n_comment from {table_name}")
    if 'r_comment' in df.columns:
        df = df.drop('r_comment')
        print(f"✓ Removed r_comment from {table_name}")
    if 'ps_comment' in df.columns:
        df = df.drop('ps_comment')
        print(f"✓ Removed ps_comment from {table_name}")
    return df

def write_single_parquet(df, output_path):
    """Write DataFrame as a single parquet file (not a directory)"""
    temp_dir = output_path + "_temp"
    df.coalesce(1).write.mode("overwrite").parquet(temp_dir)
    for file in os.listdir(temp_dir):
        if file.endswith('.parquet'):
            shutil.move(os.path.join(temp_dir, file), output_path)
            break
    shutil.rmtree(temp_dir)
    return output_path

def check_primary_keys(df, table_name, pk_columns):
    """Check for nulls in primary keys"""
    for pk in pk_columns:
        null_count = df.filter(col(pk).isNull()).count()
        if null_count > 0:
            print(f"⚠ WARNING: {table_name} has {null_count} null values in {pk}")
        else:
            print(f"✓ {table_name}.{pk} has no nulls")

def print_structure(path, prefix="", max_depth=2, current_depth=0):
    """Print directory structure"""
    if current_depth > max_depth or not os.path.exists(path):
        return
    
    items = sorted([i for i in os.listdir(path) if not i.startswith('.')])
    for i, item in enumerate(items):
        item_path = os.path.join(path, item)
        is_last = i == len(items) - 1
        
        if os.path.isfile(item_path):
            if item.endswith('.parquet'):
                size_mb = os.path.getsize(item_path) / (1024 * 1024)
                print(f"{prefix}{'└── ' if is_last else '├── '}{item} ({size_mb:.2f} MB)")
        else:
            folder_size = 0
            for root, dirs, files in os.walk(item_path):
                for file in files:
                    folder_size += os.path.getsize(os.path.join(root, file))
            folder_size_mb = folder_size / (1024 * 1024)
            folder_size_gb = folder_size_mb / 1024
            
            if folder_size_gb >= 1:
                size_str = f"{folder_size_gb:.2f} GB"
            else:
                size_str = f"{folder_size_mb:.2f} MB"
            
            print(f"{prefix}{'└── ' if is_last else '├── '}{item}/ - {size_str}")
            print_structure(item_path, prefix + ("    " if is_last else "│   "), max_depth, current_depth + 1)

# ============================================================================
# SECTION 1: LOAD ORIGINAL DATA
# ============================================================================
print("\n" + "="*80)
print("📂 LOADING ORIGINAL TPC-H DATA")
print("="*80)

try:
    orders = spark.read.parquet("../../../data/raw/tables/work_data/bases/orders_base.parquet")
    lineitem = spark.read.parquet("../../../data/raw/tables/work_data/bases/lineitem_base.parquet")
    customer = spark.read.parquet("../../../data/raw/tables/work_data/original_data/customer.parquet")
    nation = spark.read.parquet("../../../data/raw/tables/work_data/original_data/nation.parquet")
    part = spark.read.parquet("../../../data/raw/tables/work_data/original_data/part.parquet")
    partsupp = spark.read.parquet("../../../data/raw/tables/work_data/original_data/partsupp.parquet")
    region = spark.read.parquet("../../../data/raw/tables/work_data/original_data/region.parquet")
    supplier = spark.read.parquet("../../../data/raw/tables/work_data/original_data/supplier.parquet")
    print("✅ All TPC-H tables loaded successfully.")
except Exception as e:
    print(f"❌ Error loading files: {e}")
    raise

# ============================================================================
# SECTION 2: REMOVE COMMENT COLUMNS FROM ALL TABLES
# ============================================================================
print("\n" + "="*80)
print("🗑️  REMOVING COMMENT COLUMNS FROM ALL TABLES")
print("="*80)

customer = remove_comment_column(customer, "customer")
lineitem = remove_comment_column(lineitem, "lineitem")
nation = remove_comment_column(nation, "nation")
orders = remove_comment_column(orders, "orders")
part = remove_comment_column(part, "part")
partsupp = remove_comment_column(partsupp, "partsupp")
region = remove_comment_column(region, "region")
supplier = remove_comment_column(supplier, "supplier")

# ============================================================================
# SECTION 3: SUPPLIER TABLE CLEANING
# ============================================================================
print("\n" + "="*80)
print("CLEANING: SUPPLIER TABLE")
print("="*80)

supplier = supplier.drop('s_address')
print("✓ Removed s_address from supplier")

supplier = supplier.withColumn('s_acctbal', abs(col('s_acctbal')))
print("✓ Converted negative s_acctbal to positive values")

# ============================================================================
# SECTION 4: PARTSUPP TABLE - SLOWLY CHANGING DIMENSION (SCD Type 2)
# ============================================================================
print("\n" + "="*80)
print("CLEANING: PARTSUPP TABLE - SCD TYPE 2")
print("="*80)

partsupp = partsupp \
    .withColumn('valid_from', current_timestamp()) \
    .withColumn('valid_to', lit(None).cast('timestamp')) \
    .withColumn('is_current', lit(True))

print("✓ Added SCD Type 2 columns to partsupp (valid_from, valid_to, is_current)")

# ============================================================================
# SECTION 5: PART TABLE - EXTRACT IDs from p_mfgr and p_brand
# ============================================================================
print("\n" + "="*80)
print("CLEANING: PART TABLE - EXTRACT IDs")
print("="*80)

part = part.withColumn(
    'p_mfgr_id', 
    regexp_extract(col('p_mfgr'), r'#(\d+)', 1).cast(IntegerType())
)

part = part.withColumn(
    'p_brand_id', 
    regexp_extract(col('p_brand'), r'#(\d+)', 1).cast(IntegerType())
)

part = part.drop('p_mfgr', 'p_brand')

print("✓ Extracted p_mfgr_id and p_brand_id from part table")
print("✓ Removed original p_mfgr and p_brand columns")

# ============================================================================
# SECTION 6: ORDERS TABLE CLEANING
# ============================================================================
print("\n" + "="*80)
print("CLEANING: ORDERS TABLE")
print("="*80)

orders = orders.withColumn(
    'o_clerk_id',
    regexp_extract(col('o_clerk'), r'#(\d+)', 1).cast(LongType())
)

orders = orders.drop('o_clerk')
print("✓ Extracted o_clerk_id from orders table")
print("✓ Removed original o_clerk column")

orders = orders.drop('o_shippriority')
print("✓ Removed o_shippriority column (constant value)")

# ============================================================================
# SECTION 7: DATA VALIDATION & QUALITY CHECKS
# ============================================================================
print("\n" + "="*80)
print("DATA QUALITY VALIDATION")
print("="*80)

check_primary_keys(customer, "customer", ["c_custkey"])
check_primary_keys(orders, "orders", ["o_orderkey"])
check_primary_keys(part, "part", ["p_partkey"])
check_primary_keys(supplier, "supplier", ["s_suppkey"])
check_primary_keys(nation, "nation", ["n_nationkey"])
check_primary_keys(region, "region", ["r_regionkey"])

negative_acctbal = supplier.filter(col('s_acctbal') < 0).count()
if negative_acctbal == 0:
    print(f"✓ supplier.s_acctbal has no negative values (all converted to absolute)")
else:
    print(f"⚠ WARNING: {negative_acctbal} negative values still exist in s_acctbal")

# ============================================================================
# SECTION 8: SPLIT DATA BY YEAR (HISTORICAL YEARS / INCREMENTAL YEARS)
# ============================================================================
print("\n" + "="*80)
print("✂️  SPLITTING DATA BY YEAR (HISTORICAL VS INCREMENTAL)")
print("="*80)

orders_with_year = orders.withColumn("year", F.year("o_orderdate"))

distinct_years = orders_with_year.select("year").distinct().orderBy("year").collect()
all_years = [row[0] for row in distinct_years]

print(f"All years in data: {all_years}")

# Split by percentage of years (60% historical, 40% incremental)
num_years = len(all_years)
historical_year_count = int(num_years * 0.6)
historical_years = all_years[:historical_year_count]
incremental_years = all_years[historical_year_count:]

print(f"\nSplit strategy: {historical_year_count} historical years, {num_years - historical_year_count} incremental years")
print(f"Historical years (base layer): {historical_years}")
print(f"Incremental years (batches): {incremental_years}")

historical_orders = orders_with_year.filter(F.col("year").isin(historical_years)).drop("year").repartition(5)
recent_orders = orders_with_year.filter(F.col("year").isin(incremental_years)).drop("year").repartition(5)

historical_orders_count = historical_orders.count()
recent_orders_count = recent_orders.count()
total_orders_count = historical_orders_count + recent_orders_count

print(f"\nOrders distribution:")
print(f"  Historical orders: {historical_orders_count:,} ({historical_orders_count/total_orders_count*100:.1f}%)")
print(f"  Incremental orders: {recent_orders_count:,} ({recent_orders_count/total_orders_count*100:.1f}%)")

historical_orders.createOrReplaceTempView("temp_orders_hist")
recent_orders.createOrReplaceTempView("temp_orders_recent")
lineitem.createOrReplaceTempView("temp_lineitem_all")

historical_lineitem = spark.sql("""
    SELECT l.* FROM temp_lineitem_all l
    INNER JOIN temp_orders_hist o ON l.l_orderkey = o.o_orderkey
""").repartition(5)

recent_lineitem = spark.sql("""
    SELECT l.* FROM temp_lineitem_all l
    INNER JOIN temp_orders_recent o ON l.l_orderkey = o.o_orderkey
""").repartition(5)

hist_lineitem_count = historical_lineitem.count()
recent_lineitem_count = recent_lineitem.count()
total_lineitems = hist_lineitem_count + recent_lineitem_count

print(f"\nLineitem distribution:")
print(f"  Historical lineitems: {hist_lineitem_count:,} ({hist_lineitem_count/total_lineitems*100:.1f}%)")
print(f"  Incremental lineitems: {recent_lineitem_count:,} ({recent_lineitem_count/total_lineitems*100:.1f}%)")

# ============================================================================
# SECTION 9: SAVE TO SILVER LAYER AS SINGLE FILES
# ============================================================================
print("\n" + "="*80)
print("💾 SAVING DATA TO SILVER LAYER")
print("="*80)

silver_path = "../../../data/silver"
base_output_path = os.path.join(silver_path, "bases")
increments_output_path = os.path.join(silver_path, "increments")
dimensions_output_path = os.path.join(silver_path, "dimensions")

if os.path.exists(silver_path):
    print(f"\n⚠️ Warning: {silver_path} already exists.")
    response = input("Do you want to overwrite it? (yes/no): ")
    if response.lower() != 'yes':
        print("❌ Operation cancelled.")
        spark.stop()
        exit()
    shutil.rmtree(silver_path)

os.makedirs(base_output_path, exist_ok=True)
os.makedirs(increments_output_path, exist_ok=True)
os.makedirs(dimensions_output_path, exist_ok=True)

print(f"\n📁 Created silver layer structure:")
print(f"   {base_output_path}")
print(f"   {increments_output_path}")
print(f"   {dimensions_output_path}")

# Save historical data (base layer)
print("\n📦 Saving historical base layer...")
historical_orders_output = os.path.join(base_output_path, "orders_base.parquet")
write_single_parquet(historical_orders, historical_orders_output)
print(f"   ✅ Saved: orders_base.parquet ({historical_orders_count:,} records)")

historical_lineitem_output = os.path.join(base_output_path, "lineitem_base.parquet")
write_single_parquet(historical_lineitem, historical_lineitem_output)
print(f"   ✅ Saved: lineitem_base.parquet ({hist_lineitem_count:,} records)")

# ============================================================================
# SECTION 10: CREATE INCREMENTAL BATCHES - ONE BATCH PER YEAR
# ============================================================================
print("\n" + "="*80)
print("📊 CREATING INCREMENTAL BATCHES (ONE BATCH PER YEAR)")
print("="*80)

recent_orders_with_year = recent_orders.withColumn("year", F.year("o_orderdate"))
batch_stats = []

for year in incremental_years:
    print(f"\n📁 Batch for Year {year}:")
    
    batch_orders = recent_orders_with_year.filter(F.col("year") == year).drop("year").repartition(5)
    batch_orders_count = batch_orders.count()
    print(f"    Orders: {batch_orders_count:,}")
    
    batch_lineitem = recent_lineitem.join(
        batch_orders.select("o_orderkey"), 
        recent_lineitem.l_orderkey == batch_orders.o_orderkey, 
        "inner"
    ).select(recent_lineitem["*"]).repartition(5)
    
    batch_lineitem_count = batch_lineitem.count()
    print(f"    Lineitems: {batch_lineitem_count:,}")
    
    batch_folder = os.path.join(increments_output_path, f"batch_{year}")
    os.makedirs(batch_folder, exist_ok=True)
    
    orders_output = os.path.join(batch_folder, "orders.parquet")
    lineitem_output = os.path.join(batch_folder, "lineitem.parquet")
    
    write_single_parquet(batch_orders, orders_output)
    write_single_parquet(batch_lineitem, lineitem_output)
    
    orders_size_mb = os.path.getsize(orders_output) / (1024 * 1024)
    lineitem_size_mb = os.path.getsize(lineitem_output) / (1024 * 1024)
    
    print(f"    ✅ Saved: orders.parquet ({orders_size_mb:.2f} MB)")
    print(f"    ✅ Saved: lineitem.parquet ({lineitem_size_mb:.2f} MB)")
    
    batch_stats.append({
        "year": year,
        "orders": batch_orders_count,
        "lineitems": batch_lineitem_count,
        "orders_size_mb": orders_size_mb,
        "lineitem_size_mb": lineitem_size_mb
    })

# ============================================================================
# SECTION 11: SAVE CLEANED DIMENSION TABLES
# ============================================================================
print("\n" + "="*80)
print("📦 SAVING CLEANED DIMENSION TABLES TO SILVER LAYER")
print("="*80)

dimension_tables = {
    "customer": customer,
    "nation": nation,
    "part": part,
    "partsupp": partsupp,
    "region": region,
    "supplier": supplier
}

for table_name, table_df in dimension_tables.items():
    output_file = os.path.join(dimensions_output_path, f"{table_name}.parquet")
    record_count = table_df.count()
    write_single_parquet(table_df, output_file)
    file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
    print(f"   ✅ Saved: {table_name}.parquet ({record_count:,} records, {file_size_mb:.2f} MB)")

# ============================================================================
# SECTION 12: VERIFY DATA INTEGRITY
# ============================================================================
print("\n" + "="*80)
print("🔍 VERIFYING DATA INTEGRITY")
print("="*80)

verified_historical_orders = spark.read.parquet(historical_orders_output).count()
verified_historical_lineitem = spark.read.parquet(historical_lineitem_output).count()

recent_orders_total = sum(batch['orders'] for batch in batch_stats)
recent_lineitem_total = sum(batch['lineitems'] for batch in batch_stats)

print(f"\nOrders:")
print(f"  Historical (years {historical_years}): {verified_historical_orders:,}")
print(f"  Incremental (years {incremental_years}): {recent_orders_total:,}")
print(f"  Total: {verified_historical_orders + recent_orders_total:,}")
print(f"  Original total: {total_orders_count:,}")
print(f"  ✅ Match: {verified_historical_orders + recent_orders_total == total_orders_count}")

print(f"\nLineitems:")
print(f"  Historical: {verified_historical_lineitem:,}")
print(f"  Incremental: {recent_lineitem_total:,}")
print(f"  Total: {verified_historical_lineitem + recent_lineitem_total:,}")
print(f"  Original total: {total_lineitems:,}")
print(f"  ✅ Match: {verified_historical_lineitem + recent_lineitem_total == total_lineitems}")

# ============================================================================
# SECTION 13: FINAL SUMMARY
# ============================================================================
print("\n" + "="*80)
print("✨ SILVER LAYER PIPELINE COMPLETE!")
print("="*80)

print("\n📁 FINAL SILVER LAYER STRUCTURE:")
print("="*80)
print_structure(silver_path, max_depth=3)

print("\n" + "="*80)
print("📊 PIPELINE SUMMARY")
print("="*80)
print(f"✅ Historical base layer (years {historical_years}):")
print(f"   - {base_output_path}/orders_base.parquet: {verified_historical_orders:,} records")
print(f"   - {base_output_path}/lineitem_base.parquet: {verified_historical_lineitem:,} records")
print(f"\n✅ Incremental batches (one per year):")
for batch in batch_stats:
    print(f"   - Year {batch['year']}: {batch['orders']:,} orders, {batch['lineitems']:,} lineitems")
print(f"\n✅ Cleaned dimension tables:")
for table_name in dimension_tables.keys():
    print(f"   - {dimensions_output_path}/{table_name}.parquet")
print(f"\n✅ Data quality checks passed:")
print(f"   - All comment columns removed")
print(f"   - Supplier negative acctbal fixed")
print(f"   - Part IDs extracted (p_mfgr_id, p_brand_id)")
print(f"   - Order clerk IDs extracted")
print(f"   - SCD Type 2 added to partsupp")
print(f"   - Primary key null checks passed")
print("\n🎯 Silver layer is ready for consumption!")
print("   ✨ Historical years serve as the base layer")
print("   ✨ Each incremental year is its own batch")
print("   ✨ All files are single parquet files")
print("   ✨ Dimension tables are cleaned and ready")
print("="*80)

spark.stop()
print("\n✅ Spark session stopped. Pipeline completed successfully!")


📂 LOADING ORIGINAL TPC-H DATA
✅ All TPC-H tables loaded successfully.

🗑️  REMOVING COMMENT COLUMNS FROM ALL TABLES
✓ Removed c_comment from customer
✓ Removed l_comment from lineitem
✓ Removed n_comment from nation
✓ Removed o_comment from orders
✓ Removed p_comment from part
✓ Removed ps_comment from partsupp
✓ Removed r_comment from region
✓ Removed s_comment from supplier

CLEANING: SUPPLIER TABLE
✓ Removed s_address from supplier
✓ Converted negative s_acctbal to positive values

CLEANING: PARTSUPP TABLE - SCD TYPE 2
✓ Added SCD Type 2 columns to partsupp (valid_from, valid_to, is_current)

CLEANING: PART TABLE - EXTRACT IDs
✓ Extracted p_mfgr_id and p_brand_id from part table
✓ Removed original p_mfgr and p_brand columns

CLEANING: ORDERS TABLE
✓ Extracted o_clerk_id from orders table
✓ Removed original o_clerk column
✓ Removed o_shippriority column (constant value)

DATA QUALITY VALIDATION
✓ customer.c_custkey has no nulls
✓ orders.o_orderkey has no nulls
✓ part.p_partkey has n